Simulación Integral: Centro de Distribución y Ventas
Este cuaderno (Notebook) aborda de manera unificada los 5 temas críticos del modelado de sistemas:
1. **Análisis de colas** (Tiempos de espera, servicio).
2. **Procesos de inventario** (Niveles de stock y políticas de revisión).
3. **Representación de sistemas** (Capacidad y utilización).
4. **Construcción de modelos de eventos discretos** (Usando `SimPy`).
5. **Evaluación de métricas operacionales** (Réplicas múltiples e Intervalos de Confianza).

**Escenario de la Vida Real:** Simularemos una tienda de repuestos automotrices.
- Los clientes llegan aleatoriamente haciendo fila (Cola).
- Los vendedores atienden la solicitud (Servicio).
- Si el repuesto está en stock, se vende y el inventario baja (Inventario).
- Cuando el inventario cae a un punto crítico $s$, se solicita al proveedor una cantidad $Q$ que tarda cierto tiempo (Lead Time) en llegar.



In [ ]:
%pip install -r requirements.txt

  Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp314-cp314-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp314-cp314-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/12.6 MB 2.7 MB/s eta 0:00:05
   ------- -------------------------------- 2.4/12.6 MB 6.9 MB/s eta 0:00:02
   ------------------------------- -------- 10.0/12.6 MB 19.1 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 22.7 MB/s  0:00:00
Using cached pandas-3.0.3-cp314-cp314-win_amd64.whl (9.9 MB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:-


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import simpy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

# Fijamos una semilla para reproducibilidad inicial (opcional)
np.random.seed(30)
random.seed(30)

# Configuración de gráficos
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 5)


## 1. Definición de Parámetros del Sistema
Aquí definimos las variables críticas abordadas en los fundamentos matemáticos ($\lambda, \mu, c, s, Q$).


In [ ]:
# --- Parámetros de Colas (Kendall: M/M/c) ---
LAMBDA = 5.0      # Tasa de llegada: 5 clientes por hora
MU = 6.0          # Tasa de servicio: Cada vendedor atiende a 6 clientes por hora
SERVIDORES = 2    # Cantidad de vendedores ('c')

# --- Parámetros de Inventario (Política Continua s, Q) ---
STOCK_INICIAL = 100
PUNTO_REORDEN_s = 20    # Cuando quedan 20 unidades, se pide más
CANTIDAD_PEDIDO_Q = 80  # Cantidad fija a pedir al proveedor
LEAD_TIME = 24          # El proveedor tarda 24 horas en entregar

# --- Parámetros de Simulación ---
TIEMPO_SIMULACION = 24 * 30  # Simular 30 días continuos (en horas)


## 2. Construcción del Modelo de Eventos Discretos (DES)
Creamos las clases que gobernarán los eventos del sistema usando `SimPy`.
El reloj avanzará de evento a evento (llegada, inicio de atención, fin de atención, llegada de pedido).


In [ ]:
class Distribuidora:
    def __init__(self, env, num_vendedores, stock_ini, punto_reorden, cant_pedido, lead_time):
        self.env = env
        # Recurso: Representa a los servidores (vendedores)
        self.vendedores = simpy.Resource(env, capacity=num_vendedores)
        # Contenedor: Representa el inventario físico
        self.inventario = simpy.Container(env, init=stock_ini, capacity=1000)

        self.punto_reorden = punto_reorden
        self.cant_pedido = cant_pedido
        self.lead_time = lead_time

        # Estado lógico
        self.pedidos_pendientes = 0

        # Listas para guardar las métricas de evaluación
        self.tiempos_espera = []
        self.registro_inventario = []
        self.registro_tiempo_inv = []
        self.clientes_atendidos = 0
        self.ventas_perdidas = 0

    def solicitar_proveedor(self):
        """Proceso que simula el tiempo de espera del proveedor (Lead Time)."""
        self.pedidos_pendientes += 1
        # El tiempo de entrega puede variar ligeramente (ej. +/- 2 horas)
        tiempo_entrega_real = max(1, random.normalvariate(self.lead_time, 2))
        yield self.env.timeout(tiempo_entrega_real)

        # El proveedor entrega el pedido y sube el stock
        yield self.inventario.put(self.cant_pedido)
        self.pedidos_pendientes -= 1

    def controlar_inventario(self):
        """Proceso continuo que evalúa el nivel de stock (Política s, Q)."""
        while True:
            # Registrar datos para gráficos
            self.registro_inventario.append(self.inventario.level)
            self.registro_tiempo_inv.append(self.env.now)

            # Condición de Punto de Reorden (ROP)
            if self.inventario.level <= self.punto_reorden and self.pedidos_pendientes == 0:
                self.env.process(self.solicitar_proveedor())

            # Revisión periódica (cada 1 hora)
            yield self.env.timeout(1)

def cliente(env, nombre, dist, mu):
    """Simula el comportamiento de un cliente desde que llega hasta que sale."""
    llegada = env.now

    # 1. Pide un vendedor
    with dist.vendedores.request() as peticion:
        yield peticion

        # Calculamos Wq (Tiempo de espera en cola)
        tiempo_espera = env.now - llegada
        dist.tiempos_espera.append(tiempo_espera)

        # 2. Recibe el servicio (tiempo exponencial)
        tiempo_servicio = random.expovariate(mu)
        yield env.timeout(tiempo_servicio)

        # 3. Compra el inventario (1 unidad por cliente para simplificar)
        if dist.inventario.level > 0:
            yield dist.inventario.get(1)
            dist.clientes_atendidos += 1
        else:
            dist.ventas_perdidas += 1

def generador_clientes(env, dist, lam, mu):
    """Genera la llegada de clientes siguiendo un proceso de Poisson."""
    i = 0
    while True:
        # Tiempo entre llegadas exponencial
        yield env.timeout(random.expovariate(lam))
        i += 1
        env.process(cliente(env, f'Cliente {i}', dist, mu))


## 3. Ejecución y Visualización Inicial
Correremos el modelo una vez para observar la caída del inventario ("Dientes de sierra") y analizar cómo interactúan las colas y los reabastecimientos.


In [ ]:
def ejecutar_simulacion_visual(tiempo_sim):
    env = simpy.Environment()
    distribuidora = Distribuidora(env, SERVIDORES, STOCK_INICIAL, PUNTO_REORDEN_s, CANTIDAD_PEDIDO_Q, LEAD_TIME)

    # Activar procesos
    env.process(generador_clientes(env, distribuidora, LAMBDA, MU))
    env.process(distribuidora.controlar_inventario())

    # Correr
    env.run(until=tiempo_sim)

    # --- Gráfico de Inventario ---
    plt.figure(figsize=(12, 5))
    plt.step(distribuidora.registro_tiempo_inv, distribuidora.registro_inventario, where='post', color='b', label='Nivel de Inventario')
    plt.axhline(y=PUNTO_REORDEN_s, color='r', linestyle='--', label='Punto de Reorden (s)')
    plt.axhline(y=0, color='k', linestyle='-', alpha=0.5)
    plt.title('Dinámica del Inventario a lo largo del tiempo (Dientes de Sierra)')
    plt.xlabel('Tiempo (Horas)')
    plt.ylabel('Cantidad en Stock')
    plt.legend()
    plt.show()

    # --- Métricas Básicas ---
    wq_promedio = np.mean(distribuidora.tiempos_espera) * 60 # en minutos
    print(f"--- RESULTADOS DE UNA CORRIDA (DURACIÓN: {tiempo_sim} HORAS) ---")
    print(f"Clientes totales atendidos: {distribuidora.clientes_atendidos}")
    print(f"Ventas perdidas por falta de stock: {distribuidora.ventas_perdidas}")
    print(f"Tiempo promedio de espera en cola (Wq): {wq_promedio:.2f} minutos")

# Ejecutar visualmente
ejecutar_simulacion_visual(TIEMPO_SIMULACION)


## 4. Evaluación de Métricas Operacionales e Inferencia Estadística
Como los modelos de eventos discretos contienen aleatoriedad, **correr el modelo 1 sola vez no es suficiente para tomar decisiones gerenciales**.
Generaremos múltiples réplicas (ej. $N=30$) para calcular un promedio robusto y su Intervalo de Confianza del 95% para el tiempo de espera ($W_q$).


In [ ]:
def calcular_intervalo_confianza(datos, confianza=0.95):
    n = len(datos)
    media = np.mean(datos)
    error_estandar = st.sem(datos)
    # Valor de T de Student
    h = error_estandar * st.t.ppf((1 + confianza) / 2., n-1)
    return media, media - h, media + h

def evaluacion_operacional(replicas, tiempo_sim):
    resultados_wq = []
    resultados_perdidas = []

    for r in range(replicas):
        env = simpy.Environment()
        distribuidora = Distribuidora(env, SERVIDORES, STOCK_INICIAL, PUNTO_REORDEN_s, CANTIDAD_PEDIDO_Q, LEAD_TIME)

        env.process(generador_clientes(env, distribuidora, LAMBDA, MU))
        env.process(distribuidora.controlar_inventario())
        env.run(until=tiempo_sim)

        # Guardar tiempo promedio de espera de esta corrida (en minutos)
        wq_replica = np.mean(distribuidora.tiempos_espera) * 60
        resultados_wq.append(wq_replica)
        resultados_perdidas.append(distribuidora.ventas_perdidas)

    # Calcular estadísticas
    media_wq, ci_bajo, ci_alto = calcular_intervalo_confianza(resultados_wq)
    media_perdidas = np.mean(resultados_perdidas)

    print(f"--- RESULTADOS OPERACIONALES ({replicas} RÉPLICAS) ---")
    print(f"Promedio Estadístico de Espera en Cola (Wq): {media_wq:.2f} minutos")
    print(f"Intervalo de Confianza al 95%: [{ci_bajo:.2f} , {ci_alto:.2f}] minutos")
    print(f"Promedio de Ventas Perdidas por mes: {media_perdidas:.1f} unidades")

    # Determinar Utilización Teórica (Rho) = Lambda / (c * Mu)
    utilizacion = LAMBDA / (SERVIDORES * MU)
    print(f"Utilización teórica de los servidores (ρ): {utilizacion:.1%} (Ideal < 100%)")

# Correr 30 réplicas
evaluacion_operacional(replicas=50, tiempo_sim=TIEMPO_SIMULACION)
